In [ ]:
!pip install -q -U crewai ddgs

import crewai
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY2")

from crewai import LLM
llm = LLM(
    model = "gemini/gemini-flash-lite-latest",
    temperature = 0.3
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.9/238.9 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/

In [ ]:
from crewai.tools import tool
from ddgs import DDGS

@tool("DuckDuckGo Search")
def search_tool(query: str) -> str:
    """Search the web using DuckDuckGo. Input is a search query.
    Returns the top results with title, link and a short snippet."""
    try:
        results = DDGS().text(query, max_results=5)
        return "\n\n".join(
            f"Title: {r['title']}\nLink: {r['href']}\nSnippet: {r['body']}"
            for r in results
        )
    except Exception as e:
        return f"Search failed: {e}"

In [ ]:
from crewai import Agent, Task, Crew

async def run_crew(role, goal, instructions, text, tools = None):
  agent =  Agent(
      role = role,
      goal = goal,
      backstory = "You are an experienced " + role + ".",
      tools = tools or [],
      llm = llm,
      verbose = False
  )
  task = Task(
      description= instructions + "\n\nDetails:\n" + text,
      expected_output = "A clear, well-written response.",
      agent = agent
  )
  crew = Crew(
      agents = [agent],
      tasks = [task],
      verbose = False
  )
  result = await crew.kickoff_async()
  return result.raw


In [ ]:
from crewai.flow.flow import Flow, start, listen, or_
from crewai.flow.human_feedback import human_feedback

class ReportFlow(Flow):

    @start()
    def collect_topic(self):
        self.state["revisions"] = 0
        print("Topic:", self.state["topic"], "| Audience:", self.state["audience"])

    @listen(collect_topic)
    async def research(self):
        self.state["research"] = await run_crew(
            role="Research Analyst",
            goal="Find accurate, recent facts with sources",
            instructions="Search the web and collect 5 key facts about this topic, "
                         "each with its source link.",
            text="Topic: " + self.state["topic"],
            tools=[search_tool],
        )
        print("Research done")

    @listen(research)
    async def write_report(self):
        self.state["draft"] = await run_crew(
            role="Report Writer",
            goal="Write clear reports for a specific audience",
            instructions="Write a report of about 300 words for this audience using "
                         "ONLY the research notes. Use a title and short sections.",
            text="Audience: " + self.state["audience"]
                 + "\n\nResearch notes:\n" + self.state["research"],
        )
        print("Draft written")

    @human_feedback(
        message="Type 'approve' to publish, or describe what to change:",
        emit=["approved", "revise"],
        llm=llm,
        default_outcome="revise",
    )
    @listen(or_("write_report", "revise_report"))
    def review_draft(self):
        return self.state["draft"]

    @listen("revise")
    async def revise_report(self):
        self.state["revisions"] += 1
        feedback = self.last_human_feedback.feedback
        self.state["draft"] = await run_crew(
            role="Editor",
            goal="Improve reports based on reviewer feedback",
            instructions="Revise this report based on the reviewer's feedback. "
                         "Return the full updated report.\n\nFeedback: " + feedback,
            text=self.state["draft"],
        )
        print("Revision", self.state["revisions"], "done. Sending back for review...")

    @listen("approved")
    def publish_report(self):
        with open("final.md", "w") as f:
            f.write(self.state["draft"])
        print("Published to final.md after", self.state["revisions"], "revision(s)")

In [ ]:
flow = ReportFlow()
await flow.kickoff_async(inputs={
    "topic": "How AI agents are being used in Indian colleges",
    "audience": "college principals",
})

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: ReportFlow                                                                                               │
│  ID: f08b168c-aeaf-45c4-b436-58180e5448db                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: ReportFlow                                                                                               │
│  ID: f08b168c-aeaf-45c4-b436-58180e5448db                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: collect_topic                                                                                          │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Topic: How AI agents are being used in Indian colleges | Audience: college principals


╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: collect_topic                                                                                          │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: review_draft                                                                                           │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Published to final.md after 2 revision(s)


╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: publish_report                                                                                         │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: publish_report                                                                                         │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: ReportFlow                                                                                               │
│  ID: f08b168c-aeaf-45c4-b436-58180e5448db                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
from IPython.display import Markdown, display

with open("final.md") as f:
    display(Markdown(f.read()))

# Strategic Report: AI Integration in Higher Education

### Introduction
As college principals navigate the evolving landscape of higher education, understanding the practical applications of artificial intelligence (AI) is critical. This report outlines five key ways AI and AI agents are currently being utilized across Indian colleges and higher education institutions to transform operations and student support.

### Admission Verification
Indian universities and colleges are now integrating automated systems and AI agents with platforms like DigiLocker. This integration streamlines, verifies, and efficiently authenticates student admission credentials and documents. For example, leading institutions are utilizing these automated pipelines to cross-reference incoming mark sheets with central board databases in seconds, drastically reducing the manual processing time during peak enrollment seasons.

### Academic Support and Tutors
To enhance student success, educational institutions are implementing AI-powered tutoring agents. Trained specifically on official course syllabi, these virtual tutors provide personalized, 24/7 academic support and guidance. For instance, several engineering colleges have deployed conversational AI assistants that help students work through complex coding problems at midnight when teaching assistants are unavailable.

### Examination Evaluation
Managing large student bodies and heavy faculty-to-student ratios across thousands of colleges presents significant challenges. To address this, AI agents are being adopted to assist in scaling and evaluating both examinations and assignments.

### Faculty Constraints and Student Engagement
With millions of students and limited teachers across the country, institutions are exploring AI applications to supplement learning delivery. This includes utilizing EQ-focused (emotional intelligence) AI tools to inspire students and help bridge faculty constraints.

### Governance and Policy Redesign
According to EY India, institutional focus has shifted beyond basic student or faculty adoption. Current discussions center on institutional governance and policy redesign to ensure AI-driven tools build capabilities and bridge gaps, rather than deepen educational inequalities.

### Conclusion
Integrating these AI technologies offers college principals valuable opportunities to streamline administrative workflows, enhance student support, and address faculty shortages while maintaining a strong commitment to equitable educational access.